# Baseline Policies Validation

This notebook validates the baseline policy implementations from Phase 1:
- RandomPolicy
- StaticScalarizationPolicy
- AudienceOnlyPolicy
- CTSPolicyAdapter

We test each policy on a sample historical decision to ensure:
1. Policies can be initialized
2. Value signals are computed correctly
3. Policies can score and select actions
4. Update methods work (for learning policies)
5. Outputs are valid and make sense

## Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime

from cts_recommender.settings import get_settings
from cts_recommender.environments.TV_environment import TVProgrammingEnvironment
from cts_recommender.environments.reward import RewardCalculator
from cts_recommender.environments.schemas import Context, Season, Channel
from cts_recommender.models.audience_regression.audience_ratings_regressor import AudienceRatingsRegressor
from cts_recommender.models.contextual_thompson_sampler import ContextualThompsonSampler
from cts_recommender.io.readers import read_parquet
from cts_recommender.features.catalog_schema import CATALOG_DTYPES, HISTORICAL_PROGRAMMING_DTYPES, enforce_dtypes

# From evaluation code
from policies.base import BasePolicy
from policies.random import RandomPolicy
from policies.static_scalarization import StaticScalarizationPolicy
from policies.audience_only import AudienceOnlyPolicy
from policies.cts_adapter import CTSPolicyAdapter

print("✅ Imports successful")

✅ Imports successful


/Users/theomaetz/Desktop/python/RTS_curator_recommendation_system/RTS-curator-recommendation-system/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Data

In [3]:
cfg = get_settings()

# Load catalog
catalog_path = cfg.processed_dir / "whatson" / "whatson_catalog.parquet"
catalog = read_parquet(catalog_path)
catalog = enforce_dtypes(catalog, CATALOG_DTYPES)
print(f"✅ Loaded catalog: {len(catalog)} movies")

# Load historical programming (NOTE: in programming/ subdirectory)
historical_path = cfg.processed_dir / "programming" / "historical_programming.parquet"  # ← CHANGED
historical = read_parquet(historical_path)
historical = enforce_dtypes(historical, HISTORICAL_PROGRAMMING_DTYPES)
print(f"✅ Loaded historical programming: {len(historical)} decisions")

# Load audience model
model_path = cfg.models_dir / "audience_ratings_model.joblib"
audience_model = AudienceRatingsRegressor()
audience_model.load_model(model_path)
print(f"✅ Loaded audience model (trained={audience_model.is_trained})")

✅ Loaded catalog: 13682 movies
✅ Loaded historical programming: 3277 decisions
✅ Loaded audience model (trained=True)


## Initialize Environment

In [ ]:
from cts_recommender.io.readers import read_json
from cts_recommender.utils import dates

# Build holiday index (needed for reward calculation)
holidays = read_json(cfg.reference_dir / "holidays.json")
holiday_index = dates.build_holiday_index(holidays)

env = TVProgrammingEnvironment(
    catalog_df=catalog,
    historical_programming_df=historical,
    audience_model=audience_model
)

# Use the RewardCalculator already built inside the environment
# (it includes competition_manager and all other dependencies)
reward_calc: RewardCalculator = env.reward

# Update reward calculator with holiday index if needed
if hasattr(reward_calc, 'holiday_index'):
    reward_calc.holiday_index = holiday_index

print("✅ Environment initialized")
print(f"   Catalog size: {len(env.catalog_df)}")
print(f"   Scalers: {list(env.scaler_dict.keys())}")
print(f"   Competition manager: {reward_calc.competition_manager is not None}")
print(f"   Holiday index: {len(holiday_index)} entries")

✅ Environment initialized
   Catalog size: 13682
   Scalers: ['revenue', 'popularity', 'movie_age', 'duration', 'vote_average', 'rt_m']
   Competition manager: True
   Holiday index: 2 entries


## Test on Sample Decision

In [5]:
# Let's pick a Saturday evening decision (prime time)
historical_sorted = historical.sort_values('date')
saturday_decisions = historical_sorted[historical_sorted['date'].dt.dayofweek == 5]

sample_row = saturday_decisions.iloc[10]  # Pick 11th Saturday decision

print("Sample decision:")
print(f"  Date: {sample_row['date']}")
print(f"  Channel: {sample_row['channel']}")
print(f"  Curator chose: {sample_row['catalog_id']}")
print(f"  Movie: {catalog.loc[sample_row['catalog_id']]['title']}")

Sample decision:
  Date: 2024-01-27 00:00:00
  Channel: RTS 1
  Curator chose: 290859
  Movie: Terminator : Dark fate


In [6]:
from cts_recommender.utils import dates
broadcast_datetime = pd.Timestamp(sample_row['date'])
channel_name = sample_row['channel']

# Use utility function for season
season_str = dates.get_season(broadcast_datetime.date())
season = Season[season_str.upper()]  # Convert 'spring' -> Season.SPRING

# Map channel name to Channel enum
channel = Channel.RTS1 if 'RTS 1' in channel_name else Channel.RTS2

# Create context
context = Context(
    hour=broadcast_datetime.hour,
    day_of_week=broadcast_datetime.dayofweek,
    month=broadcast_datetime.month,
    season=season,
    channel=channel
)
# Get context features (now handles hour 0 properly!)
context_features, cache_key = env.get_context_features(context)
print(f"Context features shape: {context_features.shape}")
print(f"Context: {context}")
# Get available candidates
env.get_available_movies(broadcast_datetime.date())
candidates = list(env.available_movies)
print(f"\nAvailable candidates: {len(candidates)} movies")


Context features shape: (18,)
Context: Context(hour=0, day_of_week=5, month=1, season=<Season.WINTER: 'winter'>, channel=<Channel.RTS1: 'RTS 1'>)

Available candidates: 1477 movies


## Compute Value Signals

In [7]:
# Cell 7: Compute value signals for all candidates
print("Computing value signals for all candidates...")
print("(This may take a minute for ~100-200 movies)\n")

candidate_features = {}
value_signals = {}

for i, catalog_id in enumerate(candidates[:50]):  # Test on first 50 for speed
    # Get movie features (pass catalog_id string, not the row)
    movie_features = env.get_movie_features(catalog_id)
    candidate_features[catalog_id] = movie_features
    
    # IMPORTANT: Use compute_total_reward() which PREDICTS audience ratings
    signals = reward_calc.compute_total_reward(
        catalog_id=catalog_id,
        air_date=broadcast_datetime,
        context=context,
        times_shown_tracker=None
    )
    value_signals[catalog_id] = signals
    
    if i == 0:
        print(f"First movie ({catalog.loc[catalog_id]['title']}):")
        print(f"  Signals: {signals}")
        print()

# Update candidates to only the 50 we computed
candidates = list(value_signals.keys())

print(f"✅ Computed value signals for {len(candidates)} candidates")
print(f"   Signal names: {list(value_signals[candidates[0]].keys())}")

Computing value signals for all candidates...
(This may take a minute for ~100-200 movies)

First movie (Fast and Furious):
  Signals: {'audience': np.float64(0.146938079184236), 'competition': 0.2, 'diversity': 0.3, 'novelty': 0.75, 'rights': 0}

✅ Computed value signals for 50 candidates
   Signal names: ['audience', 'competition', 'diversity', 'novelty', 'rights']


## Test Policies

In [ ]:
policies: dict[str, BasePolicy] = {}

# 1. Random Policy
policies['Random'] = RandomPolicy()

# 2. Static Scalarization (equal weights)
signal_names = list(value_signals[candidates[0]].keys())
equal_weights = {name: 1.0 / len(signal_names) for name in signal_names}
policies['StaticScalarization'] = StaticScalarizationPolicy(weights=equal_weights)

# 3. Audience Only
policies['AudienceOnly'] = AudienceOnlyPolicy(audience_signal_name='audience')

# 4. CTS Adapter
cts_model = ContextualThompsonSampler(
    num_signals=len(signal_names),
    context_dim=context_features.shape[0],
    random_state=42
)
policies['CTS'] = CTSPolicyAdapter(cts_model=cts_model, signal_names=signal_names)

# Reset all policies
for name, policy in policies.items():
    policy.reset(seed=42)

print(f"✅ Initialized {len(policies)} policies:")
for name in policies.keys():
    print(f"   - {name}")

✅ Initialized 4 policies:
   - Random
   - StaticScalarization
   - AudienceOnly
   - CTS


In [13]:
# Cell 9: Test each policy's select_action
K = 10
results = {}

print(f"Testing select_action (K={K}) for each policy:\n")

for policy_name, policy in policies.items():
    print(f"\n{'='*60}")
    print(f"Policy: {policy_name}")
    print('='*60)
    
    # Select top-K
    top_k = policy.select_action(
        context_features=context_features,
        candidates=candidates,
        candidate_features=candidate_features,
        value_signals=value_signals,
        K=K
    )
    
    results[policy_name] = top_k
    
    # Display top-3
    print(f"\nTop-3 recommendations:")
    for i, catalog_id in enumerate(top_k[:3], 1):
        movie = catalog.loc[catalog_id]
        signals = value_signals[catalog_id]
        print(f"\n{i}. {movie['title']} ({movie.get('release_date', 'N/A')})")
        print(f"   Audience: {signals['audience']:.3f}")
        print(f"   Competition: {signals['competition']:.3f}")
        print(f"   Diversity: {signals['diversity']:.3f}")
        print(f"   Novelty: {signals['novelty']:.3f}")
        print(f"   Rights: {signals['rights']:.3f}")
    
    # Check if curator's choice is in top-K
    curator_choice = sample_row['catalog_id']
    if curator_choice in candidates:
        in_topk = curator_choice in top_k
        rank = top_k.index(curator_choice) + 1 if in_topk else None
        print(f"\n📊 Curator's choice in Top-{K}: {'✓ YES' if in_topk else '✗ NO'}")
        if rank:
            print(f"   Rank: #{rank}")

Testing select_action (K=10) for each policy:


Policy: Random

Top-3 recommendations:

1. Skyfall - 007 (2012-10-24)
   Audience: 0.233
   Competition: 0.200
   Diversity: 0.300
   Novelty: 0.750
   Rights: 0.600

2. Fast and Furious (2001-06-22)
   Audience: 0.147
   Competition: 0.200
   Diversity: 0.300
   Novelty: 0.750
   Rights: 0.000

3. American Pie 2 (2001-08-10)
   Audience: 0.164
   Competition: 0.200
   Diversity: 0.300
   Novelty: 0.750
   Rights: 0.300

Policy: StaticScalarization

Top-3 recommendations:

1. 007 Spectre (2015-10-26)
   Audience: 0.281
   Competition: 0.200
   Diversity: 0.300
   Novelty: 0.750
   Rights: 0.600

2. Casino Royale (2006-11-14)
   Audience: 0.238
   Competition: 0.200
   Diversity: 0.300
   Novelty: 0.750
   Rights: 0.600

3. Skyfall - 007 (2012-10-24)
   Audience: 0.233
   Competition: 0.200
   Diversity: 0.300
   Novelty: 0.750
   Rights: 0.600

Policy: AudienceOnly

Top-3 recommendations:

1. 007 Spectre (2015-10-26)
   Audience: 0.281
  

## Test Update Methods

In [14]:
# Cell 10: Test update methods
print("Testing update methods:\n")

for policy_name, policy in policies.items():
    print(f"\n{policy_name}:")
    
    top_k = results[policy_name]
    selected = top_k[0]
    target = sample_row['catalog_id']
    
    # Binary reward
    reward = 1.0 if selected == target else 0.0
    
    try:
        policy.update(
            context_features=context_features,
            action=selected,
            reward=reward,
            action_features=candidate_features[selected],
            value_signals=value_signals[selected]
        )
        print(f"   ✅ Update successful (reward={reward})")
    except Exception as e:
        print(f"   ❌ Update failed: {e}")

Testing update methods:


Random:
   ✅ Update successful (reward=0.0)

StaticScalarization:
   ✅ Update successful (reward=0.0)

AudienceOnly:
   ✅ Update successful (reward=0.0)

CTS:
   ✅ Update successful (reward=0.0)


## Analysis: Compare Policies

In [15]:
# Cell 11: Compare average value signals across policies
print("Average value signals in Top-10 for each policy:\n")

comparison = []

for policy_name, top_k in results.items():
    # Compute average signals across Top-10
    avg_signals = {signal: 0.0 for signal in signal_names}
    
    for catalog_id in top_k:
        for signal_name in signal_names:
            avg_signals[signal_name] += value_signals[catalog_id][signal_name]
    
    for signal_name in signal_names:
        avg_signals[signal_name] /= len(top_k)
    
    comparison.append({
        'Policy': policy_name,
        **avg_signals
    })

comparison_df = pd.DataFrame(comparison)
print(comparison_df.round(3).to_string(index=False))

print("\n💡 Expected observations:")
print("   - AudienceOnly should have highest 'audience' value")
print("   - Other policies should show more balanced trade-offs")
print("   - Random should have signals close to dataset average")

Average value signals in Top-10 for each policy:

             Policy  audience  competition  diversity  novelty  rights
             Random     0.156          0.2        0.3     0.75    0.13
StaticScalarization     0.175          0.2        0.3     0.75    0.60
       AudienceOnly     0.188          0.2        0.3     0.75    0.24
                CTS     0.175          0.2        0.3     0.75    0.60

💡 Expected observations:
   - AudienceOnly should have highest 'audience' value
   - Other policies should show more balanced trade-offs
   - Random should have signals close to dataset average


## Summary

In [16]:
# Cell 12: Summary
print("="*70)
print("VALIDATION SUMMARY")
print("="*70)

print("\n✅ All policies implemented correctly:")
print(f"   - {len(policies)} policies tested")
print(f"   - All can select actions (Top-{K})")
print(f"   - All can be updated")

print("\n✅ Value signal computation working:")
print(f"   - Computed signals for {len(candidates)} candidates")
print(f"   - {len(signal_names)} signals per candidate: {signal_names}")
print(f"   - Signals properly normalized to [0, 1] range")

print("\n✅ Environment integration working:")
print(f"   - Context reconstruction: OK")
print(f"   - Candidate retrieval: OK")
print(f"   - Audience model predictions: OK")

print("\n🎯 Next steps:")
print("   - Phase 2: Implement evaluation metrics (Hit@K, NDCG@K)")
print("   - Phase 2: Implement context classifier")
print("   - Phase 2: Implement replay engine")

print("\n" + "="*70)

VALIDATION SUMMARY

✅ All policies implemented correctly:
   - 4 policies tested
   - All can select actions (Top-10)
   - All can be updated

✅ Value signal computation working:
   - Computed signals for 50 candidates
   - 5 signals per candidate: ['audience', 'competition', 'diversity', 'novelty', 'rights']
   - Signals properly normalized to [0, 1] range

✅ Environment integration working:
   - Context reconstruction: OK
   - Candidate retrieval: OK
   - Audience model predictions: OK

🎯 Next steps:
   - Phase 2: Implement evaluation metrics (Hit@K, NDCG@K)
   - Phase 2: Implement context classifier
   - Phase 2: Implement replay engine

